<a href="https://colab.research.google.com/github/Ahmedali3ff/Flyrank-internship-/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedali3ff/Flyrank-internship-/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
        precision_score,
            recall_score,
                f1_score,
                    roc_auc_score,
                        confusion_matrix
                        )

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!git clone https://github.com/Ahmedali3ff/Flyrank-internship-.git

Cloning into 'Flyrank-internship-'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 139 (delta 50), reused 101 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.85 MiB | 13.70 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [8]:
import pandas as pd

DATA_PATH = "/content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Loaded successfully.")

Dataset shape: (30000, 44)
Loaded successfully.


In [12]:
df["is_declining_label"] = (
      df["trend_direction"].astype(str).str.lower() == "down"
      ).astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

print("\nDeclining rate:",
            round(df["is_declining_label"].mean(), 4))


Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining rate: 0.5421


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 3. Leakage Audit

I audited the features used by the Week-5 Logistic Regression model.

The target `is_declining_label` is derived from `trend_direction`, so `trend_direction` was excluded from the model features.

I also excluded `trend_pct` because it directly describes the trend used to create the target.

The remaining features represent content characteristics, historical traffic, engagement, freshness, search performance, and related measurements.

One important limitation is that some recent performance features may be strongly related to the outcome. Therefore, I treat the model as decision-support and interpret the measured performance conservatively rather than claiming causal prediction.

In [14]:
# Features used for the honest validation model
# Exclude target-derived fields to avoid leakage.

feature_cols = [
    "search_volume",
        "competition",
            "cpc",
                "word_count",
                    "char_count",
                        "impressions_90d",
                            "clicks_90d",
                                "pageviews_90d",
                                    "sessions_90d",
                                        "users_90d",
                                            "engaged_sessions_90d",
                                                "ai_sessions_90d",
                                                    "scroll_events_90d",
                                                        "days_with_impressions",
                                                            "days_with_sessions",
                                                                "impressions_last_30d",
                                                                    "clicks_last_30d",
                                                                        "sessions_last_30d",
                                                                            "impressions_prev_30d",
                                                                                "clicks_prev_30d",
                                                                                    "sessions_prev_30d",
                                                                                        "content_age_days",
                                                                                            "days_since_last_update",
                                                                                                "ctr",
                                                                                                    "avg_position",
                                                                                                        "engagement_rate",
                                                                                                            "scroll_rate",
                                                                                                                "ai_traffic_pct"
                                                                                                                ]

target_col = "is_declining_label"

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Number of features:", len(feature_cols))
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nExcluded potential leakage columns:")
print(["trend_direction", "trend_pct"])

Number of features: 28
X shape: (30000, 28)
y shape: (30000,)

Excluded potential leakage columns:
['trend_direction', 'trend_pct']


In [16]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
        test_size=0.20,
            random_state=42
            )

train_idx, test_idx = next(
                gss.split(X, y, groups=groups)
                )

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("Client overlap:", len(train_clients.intersection(test_clients)))

print("\nTrain declining rate:", round(y_train.mean(), 4))
print("Test declining rate:", round(y_test.mean(), 4))

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0

Train declining rate: 0.5501
Test declining rate: 0.511


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer(
    transformers=[
            (
                        "num",
                                    Pipeline([
                                                    ("imputer", SimpleImputer(strategy="median")),
                                                                    ("scaler", StandardScaler())
                                                                                ]),
                                                                                            feature_cols
                                                                                                    )
                                                                                                        ]
                                                                                                        )

honest_model = Pipeline([
                                                                                                            ("preprocessor", preprocessor),
                                                                                                                ("model", LogisticRegression(
                                                                                                                        max_iter=1000,
                                                                                                                                random_state=42
                                                                                                                                    ))
                                                                                                                                    ])

honest_model.fit(X_train, y_train)

print("Honest grouped Logistic Regression trained successfully.")

Honest grouped Logistic Regression trained successfully.


In [18]:
y_pred_honest = honest_model.predict(X_test)
y_prob_honest = honest_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully.")
print("Number of predictions:", len(y_pred_honest))

Predictions generated successfully.
Number of predictions: 6163


In [21]:
from sklearn.metrics import (
      accuracy_score,
          precision_score,
              recall_score,
                  f1_score,
                      roc_auc_score,
                          confusion_matrix
                          )

honest_accuracy = accuracy_score(y_test, y_pred_honest)
honest_precision = precision_score(y_test, y_pred_honest)
honest_recall = recall_score(y_test, y_pred_honest)
honest_f1 = f1_score(y_test, y_pred_honest)
honest_auc = roc_auc_score(y_test, y_prob_honest)

print("HONEST GROUPED VALIDATION RESULTS")
print("----------------------------------")
print("Accuracy :", round(honest_accuracy, 4))
print("Precision:", round(honest_precision, 4))
print("Recall   :", round(honest_recall, 4))
print("F1-Score :", round(honest_f1, 4))
print("ROC-AUC  :", round(honest_auc, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_honest))


HONEST GROUPED VALIDATION RESULTS
----------------------------------
Accuracy : 0.7561
Precision: 0.8071
Recall   : 0.6869
F1-Score : 0.7422
ROC-AUC  : 0.8497

Confusion Matrix:
[[2497  517]
 [ 986 2163]]


In [23]:
comparison = pd.DataFrame({
      "Metric": [
              "Accuracy",
                      "Precision",
                              "Recall",
                                      "F1",
                                              "ROC-AUC"
                                                  ],
                                                      "Week-5 Random Split": [
                                                              0.8218,
                                                                      0.8429,
                                                                              0.8250,
                                                                                      0.8339,
                                                                                              0.9124
                                                                                                  ],
                                                                                                      "Week-6 Client-Grouped Split": [
                                                                                                              honest_accuracy,
                                                                                                                      honest_precision,
                                                                                                                              honest_recall,
                                                                                                                                      honest_f1,
                                                                                                                                              honest_auc
                                                                                                                                                  ]
                                                                                                                                                  })
comparison


,Metric,Week-5 Random Split,Week-6 Client-Grouped Split
0,Accuracy,0.8218,0.756125
1,Precision,0.8429,0.807090
2,Recall,0.8250,0.686885
3,F1,0.8339,0.742151
4,ROC-AUC,0.9124,0.849691


In [31]:
# Check whether any model feature is directly derived from the target
leakage_candidates = [
    "trend_direction",
        "trend_pct",
            "is_declining_label"
            ]

print("Leakage candidate columns:")
for col in leakage_candidates:
    print(f"- {col}: {'PRESENT' if col in feature_cols else 'EXCLUDED'}")
print("\nFeatures used by the model:")
for i, col in enumerate(feature_cols, start=1):
    print(f"{i}. {col}")

Leakage candidate columns:
- trend_direction: EXCLUDED
- trend_pct: EXCLUDED
- is_declining_label: EXCLUDED

Features used by the model:
1. search_volume
2. competition
3. cpc
4. word_count
5. char_count
6. impressions_90d
7. clicks_90d
8. pageviews_90d
9. sessions_90d
10. users_90d
11. engaged_sessions_90d
12. ai_sessions_90d
13. scroll_events_90d
14. days_with_impressions
15. days_with_sessions
16. impressions_last_30d
17. clicks_last_30d
18. sessions_last_30d
19. impressions_prev_30d
20. clicks_prev_30d
21. sessions_prev_30d
22. content_age_days
23. days_since_last_update
24. ctr
25. avg_position
26. engagement_rate
27. scroll_rate
28. ai_traffic_pct


In [32]:
# Build a test-set dataframe with actual and predicted labels
error_df = df.iloc[test_idx].copy()

error_df["actual"] = y_test.values
error_df["predicted"] = y_pred_honest

error_df["error_type"] = np.where(
    (error_df["actual"] == 0) & (error_df["predicted"] == 1),
    "False Positive",
    np.where(
        (error_df["actual"] == 1) & (error_df["predicted"] == 0),
        "False Negative",
        "Correct"
    )
)

# Keep only real model errors
errors = error_df[
    error_df["error_type"] != "Correct"
].copy()

print("Total test examples:", len(error_df))
print("Total errors:", len(errors))

print("\nError counts:")
print(errors["error_type"].value_counts())

print("\nExample errors:")
display(
    errors[
        [
            "content_id",
            "client_id",
            "avg_position",
            "content_age_days",
            "impressions_last_30d",
            "sessions_last_30d",
            "trend_direction",
            "actual",
            "predicted",
            "error_type"
        ]
    ].head(10)
)

Total test examples: 6163
Total errors: 1503

Error counts:
error_type
False Negative    986
False Positive    517
Name: count, dtype: int64

Example errors:


,content_id,client_id,avg_position,content_age_days,impressions_last_30d,sessions_last_30d,trend_direction,actual,predicted,error_type
13,content_a5a2fbc76336,client_8527a891e2,39.8,238,85,3,stable,0,1,False Positive
23,content_2da6ae9d0882,client_e629fa6598,13.9,502,0,3,down,1,0,False Negative
36,content_bce275871a25,client_f369cb89fc,5.4,187,77,1,stable,0,1,False Positive
39,content_4595e8704e07,client_8527a891e2,36.3,348,0,0,down,1,0,False Negative
43,content_1938955b34c4,client_f369cb89fc,2.9,138,3,0,down,1,0,False Negative
47,content_40cb4af260c0,client_f369cb89fc,25.5,126,1,1,down,1,0,False Negative
49,content_f0717373e86e,client_8527a891e2,10.1,174,0,0,down,1,0,False Negative
51,content_d8a23b5e10c5,client_f369cb89fc,7.5,126,0,1,down,1,0,False Negative
54,content_ff8ea1364b59,client_e629fa6598,10.7,502,3,0,down,1,0,False Negative
58,content_caff51984338,client_e629fa6598,9.1,494,5,0,down,1,0,False Negative


## Failure Examples

The honest client-grouped evaluation produced 1,503 errors out of 6,163 test examples.

There were 986 false negatives and 517 false positives.

The false negatives show cases where the model predicted that content was not declining even though the measured label was "down". Several examples had very low recent sessions or impressions, suggesting that weak recent traffic can still be difficult for the model to classify correctly.

The false positives show cases where the model predicted decline even though the measured label was "stable". For example, some stable pages had relatively low recent traffic and poor average positions, which may make them look similar to declining pages based on the available features.

These examples show that the model is useful as decision-support, but its predictions should not be treated as definitive classifications without additional review.

## 4. Claim Rewrite

### Original claim

The Logistic Regression model accurately predicts which content is declining.

### Revised claim

The Logistic Regression model achieved measured performance of 0.756 accuracy, 0.742 F1, and 0.850 ROC-AUC under a client-grouped validation split.

The results are lower than the Week-5 random-split evaluation, where the model achieved an F1 score of 0.834 and ROC-AUC of 0.912.

This difference suggests that the random split may have provided a more optimistic estimate of generalization. The grouped evaluation provides a more conservative view because content from the same client is kept within a single split.

Therefore, the model should be described as directional decision-support for identifying potentially declining content, rather than as a universally accurate predictor.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.